In [ ]:
from src import config as project_config
from src.config import *

try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive/recsys")
    source = drive_root / "data" / "listens.parquet"
    if not source.is_file():
        raise FileNotFoundError(f"Загрузите listens.parquet в {source.parent}")

    local_data = Path("/content/recsys/data")
    local_data.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, local_data / source.name)
    (drive_root / "artifacts").mkdir(parents=True, exist_ok=True)
    os.environ["RECSYS_DATA_DIR"] = str(local_data)
    os.environ["RECSYS_ARTIFACTS_DIR"] = str(drive_root / "artifacts")
    project_config.DATA_DIR = local_data
    project_config.ARTIFACTS_DIR = drive_root / "artifacts"

from src.config import DATA_DIR, ARTIFACTS_DIR, device
from src.prepare_data import load_yambda_data, train_val_test_split
from src.vocabulary import Vocabulary
from src.dataset import RecDataset
from src.train import plot_loss, train_loop

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Данные: {DATA_DIR}")
print(f"Результаты: {ARTIFACTS_DIR}")

device

In [ ]:
torch.manual_seed(TRAIN_PARAMS["seed"])
np.random.seed(TRAIN_PARAMS["seed"])

run_dir = ARTIFACTS_DIR / f"baseline_t4_{TRAIN_PARAMS['data_size']}"
run_dir.mkdir(parents=True, exist_ok=True)

df = load_yambda_data(size=TRAIN_PARAMS["data_size"])
train, val, test = train_val_test_split(df)

vocab = Vocabulary(train)
with open(run_dir / "vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
with open(run_dir / "experiment_config.json", "w") as f:
    json.dump({"training": TRAIN_PARAMS, "model": GPT2_PARAMS, "split": train.attrs}, f, indent=2)

seq_len = GPT2_PARAMS["n_positions"]
train_ds = RecDataset(train, vocab, seq_len=seq_len)
val_ds = RecDataset(val, vocab, seq_len=seq_len, context=train)

print(f"Train users: {len(train)}, vocabulary: {len(vocab)}")
print(f"Train windows: {len(train_ds)}, validation windows: {len(val_ds)}")
print(f"Результаты: {run_dir}")

In [ ]:
config = GPT2Config(
    **GPT2_PARAMS,
    vocab_size=len(vocab),
    pad_token_id=vocab.pad_idx,
)
model = GPT2LMHeadModel(config).to(device)

optimizer = AdamW(
    model.parameters(),
    lr=TRAIN_PARAMS["lr"],
    weight_decay=TRAIN_PARAMS["weight_decay"],
)
train_loader = DataLoader(train_ds, batch_size=TRAIN_PARAMS["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=TRAIN_PARAMS["batch_size"], shuffle=False)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Train batches per epoch: {len(train_loader)}")

In [ ]:
loss_history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=TRAIN_PARAMS["num_epochs"],
    checkpoint_dir=run_dir / "checkpoints",
    plot_every=TRAIN_PARAMS["plot_every"],
)